# 01 — Data Exploration

Explore raw EEG signals from the Sleep-EDF and EEGMMIDB datasets.

**What this notebook covers:**
- Loading a raw EDF file with MNE
- Visualising multi-channel EEG traces
- Plotting Power Spectral Density (PSD)
- Understanding sleep stage distributions
- Checking sampling rates and channel layouts

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import mne

from src.data.loader import EEGDataLoader
from src.utils.visualization import plot_eeg_traces, plot_psd, plot_spectrogram, plot_band_power_over_time

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

## 1. Load a Sleep-EDF Subject

In [ ]:
loader = EEGDataLoader(data_dir='../data/raw/sleep-edf')

# Load subject 0 (change index to explore others)
subject = loader.load_sleep_edf(subject_id=0, data_dir='../data/raw/sleep-edf')

data   = subject['data']       # (n_channels, n_samples)
labels = subject['labels']     # sleep stage per 30-second epoch
sfreq  = subject['sfreq']
ch_names = subject['ch_names']

print(f'Data shape  : {data.shape}')
print(f'Sfreq       : {sfreq} Hz')
print(f'Duration    : {data.shape[1]/sfreq/3600:.2f} hours')
print(f'Channels    : {ch_names}')
print(f'Sleep stages: {dict(zip(*np.unique(labels, return_counts=True)))}')

## 2. Visualise Raw EEG Traces

In [ ]:
# Plot the first 30 seconds
plot_eeg_traces(
    data,
    sfreq=sfreq,
    ch_names=ch_names,
    duration=30.0,
    title='Raw EEG — First 30 Seconds'
)

## 3. Power Spectral Density

In [ ]:
plot_psd(
    data,
    sfreq=sfreq,
    ch_names=ch_names,
    fmax=40.0,
    title='PSD — Full Recording'
)

## 4. Spectrogram — Time-Frequency View

In [ ]:
# Channel 0 (usually EEG Fpz-Cz in Sleep-EDF)
plot_spectrogram(
    data, sfreq=sfreq, channel=0,
    title=f'Spectrogram — {ch_names[0]}'
)

## 5. Sleep Stage Distribution

In [ ]:
stage_names = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}
valid = labels[labels >= 0]
counts = np.bincount(valid, minlength=5)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    [stage_names[i] for i in range(5)],
    counts / counts.sum() * 100,
    color=['#e74c3c','#3498db','#2ecc71','#9b59b6','#f39c12']
)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{c} epochs', ha='center', fontsize=9)
ax.set_ylabel('Percentage (%)')
ax.set_title('Sleep Stage Distribution')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Band Power Over Time (Hypnogram Proxy)

In [ ]:
# Use a short window to track how band power shifts through the night
plot_band_power_over_time(
    data, sfreq=sfreq,
    window_sec=30.0,
    title='EEG Band Power Over Night'
)